# 📄 Document Extraction & Foundry Summarization Pipeline

**End-to-end document processing for insurance artifacts using Fabric AI + Foundry.**

**Fabric Features Showcased:**
- Azure AI Document Intelligence — extract text, tables, key-value pairs from PDFs/images
- Azure OpenAI (via Foundry Agent) — summarize extracted content
- SynapseML — distributed document processing at scale
- OneLake Shortcuts — access documents from external storage
- Delta Lake — store all results in lakehouse (medallion architecture)
- MLflow — track document processing metrics
- Fabric Workspace Identity — secure access to Azure AI services

**Architecture:**
```
Documents (PDF/Images)  →  Document Intelligence (Extract)  → 
Foundry Agent (Summarize) → Lakehouse Delta Tables
```

**Use Cases:**
- Claims documents (medical records, police reports, estimates)
- Policy documents (applications, endorsements, renewals)
- Legal documents (contracts, underwriting agreements)
- Financial documents (invoices, receipts, bank statements)

**No SparkSession import — `spark` pre-initialized by Fabric.**

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Load shared utilities                                              ║
# ╚══════════════════════════════════════════════════════════════════════╝
%run 00_fabric_native_utils

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Parameters                                                          ║
# ╚══════════════════════════════════════════════════════════════════════╝

# Document storage paths (OneLake or external via shortcuts)
RAW_DOCS_PATH = "Files/raw_documents"  # Bronze - raw PDFs/images
EXTRACTED_PATH = "Tables/bronze_document_extracted"  # Bronze - extraction results
SUMMARIZED_PATH = "Tables/silver_document_summarized"  # Silver - with summaries
INDEXED_PATH = "Tables/gold_document_index"  # Gold - searchable index

# Azure AI Services (retrieved via Workspace Identity)
DOCUMENT_INTELLIGENCE_ENDPOINT = get_secret("document-intelligence-endpoint")  # From Key Vault
DOCUMENT_INTELLIGENCE_KEY = get_secret("document-intelligence-key")
FOUNDRY_ENDPOINT = get_secret("foundry-endpoint")  # Azure AI Foundry project endpoint
FOUNDRY_AGENT_NAME = "insurance-doc-summarizer"  # Foundry agent for summarization

# Processing parameters
BATCH_SIZE = 100  # Documents per batch
MAX_PAGES_PER_DOC = 50  # Skip very large docs
ENABLE_TABLES = True  # Extract tables from documents
ENABLE_KV_PAIRS = True  # Extract key-value pairs

print("✅ Parameters configured")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 1: Document Discovery & Cataloging                         ║
# ║  Scan OneLake for documents, create Bronze catalog                  ║
# ╚══════════════════════════════════════════════════════════════════════╝

from pyspark.sql.functions import (
    input_file_name, current_timestamp, sha2, concat_ws,
    regexp_extract, split, size, col
)

def discover_documents():
    """Scan OneLake for documents and create a catalog.
    
    Supports: PDF, PNG, JPG, JPEG, TIFF, BMP
    Creates Bronze catalog with: file_path, file_size, file_hash, doc_type
    """
    print("📁 Discovering documents in OneLake...")
    
    # Use notebookutils.fs (built-in) to list files
    files = notebookutils.fs.ls(RAW_DOCS_PATH)
    
    # Convert to DataFrame
    from pyspark.sql import Row
    file_rows = [
        Row(
            file_path=f.path,
            file_name=f.name,
            file_size=f.size,
            file_modified=f.modifyTime,
        )
        for f in files if f.isFile and f.name.lower().endswith(('.pdf', '.png', '.jpg', '.jpeg', '.tiff', '.bmp'))
    ]
    
    if not file_rows:
        print("⚠️  No documents found. Upload PDFs/images to:", RAW_DOCS_PATH)
        return None
    
    df_catalog = spark.createDataFrame(file_rows)
    
    # Add metadata
    df_catalog = df_catalog \
        .withColumn("doc_type", 
            regexp_extract(col("file_name"), r"(claim|policy|invoice|medical|legal|contract)", 1)) \
        .withColumn("discovered_at", current_timestamp()) \
        .withColumn("processing_status", lit("pending"))
    
    # Save to Bronze lakehouse
    df_catalog.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("bronze_document_catalog")
    
    print(f"✅ Cataloged {df_catalog.count()} documents → bronze_document_catalog")
    return df_catalog

# Run discovery
df_docs = discover_documents()
if df_docs:
    display(df_docs.limit(10))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 2: Document Extraction with Azure AI Document Intelligence ║
# ║  Uses: SynapseML AnalyzeDocument transformer (DISTRIBUTED)          ║
# ╚══════════════════════════════════════════════════════════════════════╝

from synapse.ml.cognitive import AnalyzeDocument
from pyspark.sql.functions import col, explode, flatten

def extract_documents_distributed(df_docs):
    """Extract text, tables, and key-value pairs at scale using SynapseML.
    
    SynapseML is PRE-INSTALLED in Fabric and provides distributed AI transformers.
    AnalyzeDocument calls Azure AI Document Intelligence in parallel across executors.
    """
    print("🔍 Extracting documents with Azure AI Document Intelligence...")
    
    # Configure the AnalyzeDocument transformer
    analyzer = AnalyzeDocument() \
        .setSubscriptionKey(DOCUMENT_INTELLIGENCE_KEY) \
        .setLocation("eastus")  # or extract from endpoint \
        .setImageUrlCol("file_path") \
        .setOutputCol("analysis") \
        .setModelId("prebuilt-layout")  # or prebuilt-invoice, prebuilt-receipt, etc.
    
    # Apply transformation (distributed across Spark cluster)
    df_analyzed = analyzer.transform(df_docs)
    
    # Extract structured results
    df_extracted = df_analyzed.select(
        col("file_path"),
        col("file_name"),
        col("doc_type"),
        col("analysis.analyzeResult.content").alias("full_text"),  # All text
        col("analysis.analyzeResult.pages").alias("pages"),  # Page details
        col("analysis.analyzeResult.tables").alias("tables"),  # Extracted tables
        col("analysis.analyzeResult.keyValuePairs").alias("key_value_pairs"),  # Form fields
        current_timestamp().alias("extracted_at")
    )
    
    # Save to Bronze lakehouse
    df_extracted.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("bronze_document_extracted")
    
    print(f"✅ Extracted {df_extracted.count()} documents → bronze_document_extracted")
    return df_extracted

# Run extraction (on pending documents)
if df_docs:
    df_extracted = extract_documents_distributed(df_docs.limit(BATCH_SIZE))  # Process in batches
    display(df_extracted.select("file_name", "doc_type", "full_text").limit(5))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 3: Foundry Agent Integration for Summarization             ║
# ║  Uses: Azure AI Foundry agent for intelligent document summary      ║
# ╚══════════════════════════════════════════════════════════════════════╝

import requests
import json
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, StructType, StructField

def call_foundry_agent(text: str, doc_type: str) -> dict:
    """Call Foundry agent to summarize document text.
    
    Foundry agents provide:
    - Pre-trained LLM models (GPT-4, etc.)
    - Custom instructions/prompts
    - Tool calling capabilities
    - Built-in monitoring and evaluation
    
    Returns: {summary, key_entities, sentiment, action_items}
    """
    try:
        # Get Foundry token via Workspace Identity
        foundry_token = get_token("https://ml.azure.com")  # Foundry resource
        
        # Prepare prompt based on document type
        prompts = {
            "claim": "Summarize this insurance claim document. Extract: claim type, loss description, amount, injuries, fault determination.",
            "policy": "Summarize this insurance policy document. Extract: coverage type, limits, deductibles, effective dates, insured parties.",
            "medical": "Summarize this medical document. Extract: diagnosis, treatment, prognosis, restrictions, costs.",
            "invoice": "Summarize this invoice. Extract: vendor, amount, date, items, payment terms.",
            "legal": "Summarize this legal document. Extract: parties, terms, obligations, dates, jurisdiction.",
        }
        prompt = prompts.get(doc_type, "Provide a concise summary of this document.") + "\n\n" + text[:8000]  # Limit context
        
        # Call Foundry agent API
        response = requests.post(
            f"{FOUNDRY_ENDPOINT}/agents/{FOUNDRY_AGENT_NAME}/invoke",
            headers={
                "Authorization": f"Bearer {foundry_token}",
                "Content-Type": "application/json"
            },
            json={
                "messages": [
                    {"role": "user", "content": prompt}
                ],
                "max_tokens": 500
            },
            timeout=30
        )
        
        if response.status_code == 200:
            result = response.json()
            summary = result["choices"][0]["message"]["content"]
            return {
                "summary": summary,
                "tokens_used": result.get("usage", {}).get("total_tokens", 0),
                "model": result.get("model", "unknown")
            }
        else:
            return {"summary": f"Error: {response.status_code}", "tokens_used": 0, "model": "error"}
    
    except Exception as e:
        return {"summary": f"Exception: {str(e)}", "tokens_used": 0, "model": "error"}

# Register as UDF for distributed processing
schema = StructType([
    StructField("summary", StringType()),
    StructField("tokens_used", StringType()),
    StructField("model", StringType())
])
summarize_udf = udf(call_foundry_agent, schema)

print("✅ Foundry agent UDF registered")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 4: Apply Summarization & Create Silver Layer               ║
# ║  Combines extraction + Foundry summary → Silver lakehouse           ║
# ╚══════════════════════════════════════════════════════════════════════╝

from pyspark.sql.functions import col, length, when

def create_silver_with_summaries():
    """Apply Foundry summarization and create Silver document layer.
    
    Bronze: Raw extraction results
    Silver: Cleansed + summarized + structured metadata
    """
    print("🪙 Creating Silver layer with summaries...")
    
    # Read Bronze extracted documents
    df_bronze = spark.table("bronze_document_extracted")
    
    # Only summarize documents with sufficient text
    df_to_summarize = df_bronze \
        .filter(length(col("full_text")) > 100) \
        .filter(length(col("full_text")) < 50000)  # Skip very large docs
    
    # Apply Foundry summarization (distributed via UDF)
    df_summarized = df_to_summarize \
        .withColumn("foundry_result", summarize_udf(col("full_text"), col("doc_type"))) \
        .withColumn("summary", col("foundry_result.summary")) \
        .withColumn("tokens_used", col("foundry_result.tokens_used")) \
        .withColumn("model_used", col("foundry_result.model")) \
        .drop("foundry_result")
    
    # Add Silver layer metadata
    df_silver = df_summarized \
        .withColumn("char_count", length(col("full_text"))) \
        .withColumn("page_count", size(col("pages"))) \
        .withColumn("table_count", size(col("tables")) if ENABLE_TABLES else lit(0)) \
        .withColumn("processing_quality", 
            when(length(col("summary")) > 50, "high")
            .when(length(col("summary")) > 20, "medium")
            .otherwise("low")) \
        .withColumn("silver_created_at", current_timestamp())
    
    # Save to Silver lakehouse
    df_silver.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("silver_document_summarized")
    
    print(f"✅ Created Silver layer → silver_document_summarized")
    
    # Show statistics
    df_silver.groupBy("doc_type", "processing_quality").count().show()
    
    return df_silver

# Create Silver layer
df_silver = create_silver_with_summaries()
display(df_silver.select("file_name", "doc_type", "summary", "tokens_used").limit(10))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 5: Extract Structured Data (Tables → Gold)                 ║
# ║  Parse extracted tables into structured Gold layer                  ║
# ╚══════════════════════════════════════════════════════════════════════╝

from pyspark.sql.functions import explode, col, posexplode

def extract_tables_to_gold():
    """Extract tables from documents into structured Gold tables.
    
    Example: Invoice tables → fact_invoice_line_items
             Claim estimates → fact_claim_line_items
    """
    print("📊 Extracting tables to Gold layer...")
    
    df_silver = spark.table("silver_document_summarized")
    
    # Explode tables array
    df_tables = df_silver \
        .filter(size(col("tables")) > 0) \
        .select(
            col("file_path"),
            col("file_name"),
            col("doc_type"),
            posexplode(col("tables")).alias("table_idx", "table_data")
        ) \
        .select(
            col("file_path"),
            col("file_name"),
            col("doc_type"),
            col("table_idx"),
            col("table_data.rowCount").alias("row_count"),
            col("table_data.columnCount").alias("column_count"),
            col("table_data.cells").alias("cells")
        )
    
    # Save table metadata to Gold
    df_tables.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("gold_document_tables")
    
    print(f"✅ Extracted {df_tables.count()} tables → gold_document_tables")
    return df_tables

# Extract tables
if ENABLE_TABLES:
    df_tables_gold = extract_tables_to_gold()
    display(df_tables_gold.limit(10))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 6: Create Searchable Gold Index                            ║
# ║  Final Gold layer with full-text search and metadata               ║
# ╚══════════════════════════════════════════════════════════════════════╝

from pyspark.sql.functions import concat_ws, regexp_replace, lower, trim

def create_gold_document_index():
    """Create production-ready Gold document index.
    
    Features:
    - Searchable full text (concatenated from all fields)
    - Document metadata (type, size, pages, quality)
    - Processing lineage (extraction timestamps, models used)
    - Ready for Power BI Direct Lake
    """
    print("🏆 Creating Gold document index...")
    
    df_silver = spark.table("silver_document_summarized")
    
    df_gold = df_silver.select(
        # Document identifiers
        sha2(col("file_path"), 256).alias("document_id"),
        col("file_name"),
        col("file_path"),
        col("doc_type"),
        
        # Content & summaries
        col("full_text"),
        col("summary"),
        
        # Searchable field (normalized)
        lower(trim(regexp_replace(
            concat_ws(" ", col("full_text"), col("summary")),
            r"[^a-zA-Z0-9\s]", " "
        ))).alias("searchable_text"),
        
        # Metadata
        col("char_count"),
        col("page_count"),
        col("table_count"),
        col("processing_quality"),
        
        # AI processing details
        col("tokens_used"),
        col("model_used"),
        
        # Lineage
        col("extracted_at"),
        col("silver_created_at"),
        current_timestamp().alias("gold_created_at")
    )
    
    # Save to Gold lakehouse with OPTIMIZE
    df_gold.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("gold_document_index")
    
    # Optimize for queries (V-Order automatic in Fabric)
    optimize_table("gold_document_index")  # From utilities
    
    print(f"✅ Gold index created → gold_document_index")
    print("\n📊 Summary Statistics:")
    df_gold.describe(["char_count", "page_count", "tokens_used"]).show()
    
    return df_gold

# Create Gold index
df_gold_index = create_gold_document_index()
display(df_gold_index.select("document_id", "file_name", "doc_type", "summary", "processing_quality").limit(10))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 7: MLflow Tracking & Monitoring                            ║
# ║  Track document processing metrics in MLflow                        ║
# ╚══════════════════════════════════════════════════════════════════════╝

import mlflow

def log_processing_metrics():
    """Log document processing metrics to MLflow for monitoring.
    
    MLflow is BUILT-IN to Fabric — no setup needed.
    """
    df_gold = spark.table("gold_document_index")
    
    # Calculate metrics
    total_docs = df_gold.count()
    avg_pages = df_gold.agg({"page_count": "avg"}).collect()[0][0]
    avg_tokens = df_gold.agg({"tokens_used": "avg"}).collect()[0][0]
    quality_dist = df_gold.groupBy("processing_quality").count().collect()
    
    # Start MLflow run
    with mlflow.start_run(run_name="document_processing_batch"):
        mlflow.log_param("batch_size", BATCH_SIZE)
        mlflow.log_param("foundry_agent", FOUNDRY_AGENT_NAME)
        
        mlflow.log_metric("total_documents", total_docs)
        mlflow.log_metric("avg_pages_per_doc", avg_pages or 0)
        mlflow.log_metric("avg_tokens_per_doc", avg_tokens or 0)
        
        for row in quality_dist:
            mlflow.log_metric(f"quality_{row['processing_quality']}_count", row['count'])
        
        print("✅ Metrics logged to MLflow")
        print(f"   Total documents: {total_docs}")
        print(f"   Avg pages: {avg_pages:.1f}")
        print(f"   Avg tokens: {avg_tokens:.0f}")

# Log metrics
log_processing_metrics()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 8: Search & Retrieval Functions                            ║
# ║  Demonstrate document search capabilities                           ║
# ╚══════════════════════════════════════════════════════════════════════╝

from pyspark.sql.functions import col, expr

def search_documents(search_terms: str, doc_types: list = None, limit: int = 10):
    """Full-text search across document index.
    
    Args:
        search_terms: Keywords to search for
        doc_types: Filter by document types (e.g., ['claim', 'policy'])
        limit: Max results
    
    Returns Spark DataFrame with matching documents
    """
    df_gold = spark.table("gold_document_index")
    
    # Convert search terms to lowercase for case-insensitive search
    terms_lower = search_terms.lower()
    
    # Apply filters
    df_results = df_gold.filter(col("searchable_text").contains(terms_lower))
    
    if doc_types:
        df_results = df_results.filter(col("doc_type").isin(doc_types))
    
    # Order by relevance (documents with more matches)
    # Simple approach: count keyword occurrences
    keywords = terms_lower.split()
    for kw in keywords:
        df_results = df_results.withColumn(
            f"match_{kw}",
            expr(f"length(searchable_text) - length(regexp_replace(searchable_text, '{kw}', ''))") / len(kw)
        )
    
    match_cols = [f"match_{kw}" for kw in keywords]
    df_results = df_results.withColumn(
        "relevance_score",
        sum([col(c) for c in match_cols])
    )
    
    # Return top results
    return df_results \
        .select("document_id", "file_name", "doc_type", "summary", "relevance_score") \
        .orderBy(col("relevance_score").desc()) \
        .limit(limit)

# Example searches
print("\n🔎 Example 1: Search for 'car accident injury'")
results1 = search_documents("car accident injury", doc_types=["claim"])
display(results1)

print("\n🔎 Example 2: Search for 'policy renewal premium'")
results2 = search_documents("policy renewal premium", doc_types=["policy", "invoice"])
display(results2)

## 🎯 Summary

This notebook demonstrates **end-to-end document processing** using Fabric + Foundry:

**Bronze Layer:**
- `bronze_document_catalog` — file inventory
- `bronze_document_extracted` — raw Document Intelligence output

**Silver Layer:**
- `silver_document_summarized` — cleansed with Foundry summaries

**Gold Layer:**
- `gold_document_index` — searchable production index
- `gold_document_tables` — extracted structured tables

**Key Implementation Decisions:**
1. **Foundry for Summarization** — leverage pre-trained agents, but store results in Lakehouse
2. **SynapseML for Scale** — distributed Document Intelligence calls across Spark cluster
3. **Medallion Architecture** — Bronze (raw) → Silver (enriched) → Gold (curated)
4. **MLflow Tracking** — monitor processing metrics, token usage, quality scores
5. **Direct Lake Ready** — Gold tables optimized for Power BI semantic models

**Next Steps:**
- Integrate with Azure AI Search for vector search
- Add RAG (Retrieval Augmented Generation) for Q&A
- Build Power BI dashboard on `gold_document_index`
- Set up Fabric Activator alerts for high-value documents